# Freeform Optimization

In [ ]:
import numpy as np

from optiland import optic, optimization
from optiland.optimization import minimize

Define a starting lens:

We define a singlet lens, which has a freeform as its first surface.

The freeform surface is defined as:

$z(x, y) = \frac{r^2}{R \cdot (1 + \sqrt{(1 - (1 + k) \cdot r^2 / R^2)})} + \sum\limits_{i}\sum\limits_{j}{C_{i, j} \cdot x^i \cdot y^j}$

where $x$ and $y$ are the local surface coordinates, $r^2 = x^2 + y^2$, $R$ is the radius of curvature, $k$ is the conic constant and $C_{i, j}$ is the polynomial coefficient for indices $i, j$.

In [ ]:
lens = optic.Optic()

# add surfaces
lens.surfaces.add(index=0, thickness=np.inf)
lens.surfaces.add(
    index=1,
    radius=100,
    thickness=5,
    surface_type="polynomial",
    is_stop=True,
    material="SF11",
    coefficients=[],
)
lens.surfaces.add(index=2, thickness=30, radius=-1000)
lens.surfaces.add(index=3)

# set aperture
lens.set_aperture(aperture_type="EPD", value=15)

# add field
lens.fields.set_type(field_type="angle")
lens.fields.add(y=0)

# add wavelength
lens.wavelengths.add(value=0.55, is_primary=True)

# draw lens
lens.draw(num_rays=5)

Define optimization problem:

In [ ]:
problem = optimization.OptimizationProblem()

Add operands (targets for optimization). We will minimize the RMS spot size on-axis and force the on-axis field chief ray to intersect the image plane at y = 3 mm.

In [ ]:
# RMS spot size operand
input_data = {
    "optic": lens,
    "surface_number": -1,
    "Hx": 0,
    "Hy": 0,
    "wavelength": 0.55,
    "num_rays": 5,
}
problem.add_operand(
    operand_type="rms_spot_size",
    target=0,
    weight=1,
    input_data=input_data,
)

# Real y-intercept operand
input_data = {
    "optic": lens,
    "surface_number": -1,
    "Hx": 0,
    "Hy": 0,
    "Px": 0,
    "Py": 0,
    "wavelength": 0.55,
}
problem.add_operand(
    operand_type="real_y_intercept",
    target=3,
    weight=1,  # <-- target=3
    input_data=input_data,
)

Define variables - let the first 9 coefficients of the polynomial coefficients vary.

In [ ]:
for i in range(3):
    for j in range(3):
        problem.add_variable(
            lens,
            "polynomial_coeff",
            surface_number=1,
            coeff_index=(i, j),
        )

Check initial merit function value and system properties:

In [ ]:
problem.info()

Run optimization:

In [ ]:
result = minimize(problem, "dls")

Print result summary:

In [ ]:
print(result)

Print merit function value and system properties after optimization:

In [ ]:
problem.info()

Draw final lens:

In [ ]:
lens.draw(num_rays=5)